# LendingClub Credit Risk · Leakage-safe Experiment Notebook

복구된 원본 실험 notebook을 포트폴리오 검토용으로 정리한 버전입니다.

원본에는 초기 실험 과정에서 target이 numeric feature에 함께 들어가 **비정상적으로 1.00 성능이 나온 셀**이 남아 있었고, 이후 직접 leakage를 의심해 변수를 제거하고 다시 실험한 흔적이 있습니다.

이 cleaned notebook은 그 학습 과정을 숨기지 않되, 실행 흐름은 다음 원칙으로 정리했습니다.

- target은 feature에서 명시적으로 제외
- 원본에서 leakage-risk로 분류한 `sub_grade`, `int_rate`, `fico_avg` 제외
- train/test split 후 scaler를 train에만 fit
- SMOTE는 train에만 적용
- Recall을 주요 평가 기준으로 사용

원본 실행 출력과 환경 의존 셀은 제거했습니다.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from imblearn.over_sampling import SMOTE
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, cross_val_score, train_test_split
from sklearn.preprocessing import StandardScaler

DATA_PATH = Path("../data/accepted_df2.csv")

## 1. Load the processed analysis dataset

복구된 원본 notebook은 `accepted_df2.csv`를 입력으로 사용했습니다.  
이 파일은 원본 LendingClub raw CSV에서 전처리된 중간 분석 데이터이며 저장소에는 포함하지 않습니다.

In [ ]:
df = pd.read_csv(DATA_PATH, low_memory=False)

print(df.shape)
print(df["loan_status_binary"].value_counts())

## 2. Leakage-safe feature set

원본 notebook은 상관관계와 초기 모델 결과를 확인한 뒤 `sub_grade`, `int_rate`, `fico_avg`를 leakage-risk 변수로 분류해 제외했습니다.

또한 이 정리본에서는 원본 초기 셀과 달리 `loan_status_binary`가 X에 들어가지 않도록 명시적으로 제거합니다.

In [ ]:
TARGET = "loan_status_binary"
LEAKAGE_RISK_VARS = ["sub_grade", "int_rate", "fico_avg"]

X = df.drop(columns=[TARGET] + LEAKAGE_RISK_VARS, errors="ignore")
X = X.select_dtypes(include=["int64", "float64"]).copy()
y = df[TARGET].copy()

X = X.dropna()
y = y.loc[X.index]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.30,
    stratify=y,
    random_state=42,
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## 3. SMOTE + Logistic Regression baseline

SMOTE는 train split에만 적용합니다.

복구된 notebook의 leakage 제거 이후 기록에는 Accuracy 약 0.604, Precision 약 0.281, Recall 약 0.630, F1 약 0.389의 실험 결과가 남아 있습니다. 아래 셀은 같은 문제 설정을 더 안전한 scaling 순서로 다시 실행할 수 있게 정리했습니다.

In [ ]:
smote = SMOTE(random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train_scaled, y_train)

baseline = LogisticRegression(max_iter=1000)
baseline.fit(X_train_sm, y_train_sm)

pred = baseline.predict(X_test_scaled)

baseline_metrics = {
    "accuracy": accuracy_score(y_test, pred),
    "precision": precision_score(y_test, pred, zero_division=0),
    "recall": recall_score(y_test, pred),
    "f1": f1_score(y_test, pred),
}

baseline_metrics

In [ ]:
print(classification_report(y_test, pred, digits=4))
print(confusion_matrix(y_test, pred))

## 4. Recall-oriented tuning

원본 notebook에서 실제로 `GridSearchCV`, `RandomizedSearchCV`, manual CV, Optuna를 사용해 Logistic Regression을 Recall 기준으로 비교했습니다.

여기서는 가장 검토하기 쉬운 GridSearchCV 예시를 남깁니다.

In [ ]:
param_grid = {
    "C": [0.01, 0.1, 1, 10],
    "solver": ["liblinear", "lbfgs"],
}

grid = GridSearchCV(
    LogisticRegression(max_iter=1000),
    param_grid=param_grid,
    scoring="recall",
    cv=5,
    n_jobs=1,
)

grid.fit(X_train_scaled, y_train)
grid_pred = grid.predict(X_test_scaled)

{
    "best_params": grid.best_params_,
    "accuracy": accuracy_score(y_test, grid_pred),
    "precision": precision_score(y_test, grid_pred, zero_division=0),
    "recall": recall_score(y_test, grid_pred),
    "f1": f1_score(y_test, grid_pred),
}

## 5. Random Forest feature importance

복구된 notebook의 후반부에서 Random Forest를 사용해 feature importance를 확인한 흐름을 유지했습니다.

In [ ]:
rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1,
)
rf.fit(X_train, y_train)

rf_pred = rf.predict(X_test)
print(classification_report(y_test, rf_pred, digits=4))

feature_importance = (
    pd.Series(rf.feature_importances_, index=X_train.columns)
    .sort_values(ascending=False)
    .head(20)
)

feature_importance.sort_values().plot(
    kind="barh",
    figsize=(9, 7),
    title="Top 20 Feature Importances",
)
plt.tight_layout()
plt.show()

## What changed from the recovered notebook

- 설치 로그/에러 출력 제거
- target이 X에 들어가던 초기 실험 셀 제거
- split 전에 전체 데이터에 scaler를 fit하던 흐름을 train-only scaling으로 수정
- 실험 목적이 중복되는 셀을 축약
- 실제 분석 의도였던 **Recall 중심 평가 + 불균형 처리 + 튜닝 + 중요도 확인** 흐름은 유지

더 넓은 모델 비교와 preprocessing은 `src/modeling_pipeline.py`와 README를 참고하세요.